# MusicScope™ — Executive Notebook (24 Charts + Momentum)
## THIS IS A PROPOSED NOTEBOOK FORMAT!!! TO BE USED FOR CONSIDERATION IN BUILDING A NOTEBOOK

**How we read:** punchline-first titles → direct labels → color for signal, grey for context.  
**For recruiters/A&Rs:** automatic “DEMO DATA” banner/watermark until live sources are detected.

In [ ]:
from __future__ import annotations

# ---- Stdlib
import math, re, warnings, itertools
from dataclasses import dataclass
from typing import Optional, Iterable, Dict, List, Tuple

# ---- Data & viz
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter, AutoDateLocator
from IPython.display import HTML, display

# ---- Optional (for bar race)
try:
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False
    warnings.warn("Plotly not installed. Install with: pip install plotly")

# ---- Pandas display
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 3)

# ---- Matplotlib defaults (clean, no chartjunk)
mpl.rcParams.update({
    "figure.dpi": 140,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,  # we'll direct-label instead
})

# ---- Accessible palette (avoid red/green pairings)
GREY_3 = "#8E8E8E"   # annotation grey
GREY_2 = "#B0B0B0"   # de-emphasis grey
GREY_1 = "#D7D7D7"   # faint grid/WM
POS_C  = "#1B9E77"   # teal (positive)
NEG_C  = "#D95F02"   # orange (risk)
ACCENT = "#E7298A"   # magenta accent
BLUE_HI = "#1f77b4"  # highlight blue (pre-breakout ≥55)
PALETTE = ["#1B9E77","#7570B3","#D95F02","#E7298A","#66A61E","#E6AB02","#A6761D","#666666"]

DATE_FMT = DateFormatter("%b %d %Y")  # ALWAYS include year

def pick_color(i:int) -> str:
    return PALETTE[i % len(PALETTE)]

def slide(figsize=(11,6), watermark: Optional[str]=None):
    """Slide-like canvas; optional ‘DEMO DATA’ watermark."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_anchor("NW")
    if watermark:
        fig.text(0.5, 0.5, watermark, color=GREY_1, fontsize=60, ha="center", va="center",
                 alpha=0.35, rotation=30)
    return fig, ax

def action_title(ax: mpl.axes.Axes, finding: str, implication: str, action: str) -> None:
    """Write titles as: Finding → Why it matters → Action"""
    ax.set_title(f"{finding} → {implication} → {action}")

def direct_line_labels(ax: mpl.axes.Axes, fontsize: int = 10):
    """Put labels at end of lines; reduce legend-chasing."""
    lines = [ln for ln in ax.get_lines() if not ln.get_label().startswith("_")]
    for ln in lines:
        x, y = ln.get_xdata(), ln.get_ydata()
        if len(x)==0: continue
        ax.annotate(ln.get_label(), xy=(x[-1], y[-1]), xytext=(5,0), textcoords="offset points",
                    va="center", fontsize=fontsize, color=ln.get_color(), fontweight="bold")
    if ax.get_legend(): ax.get_legend().remove()

def label_bars(ax: mpl.axes.Axes, fmt="{:.0f}", fontsize=10):
    """Label bars directly."""
    for p in ax.patches:
        h = p.get_height()
        if h == 0: continue
        x = p.get_x()+p.get_width()/2
        y = p.get_y()+h
        ax.text(x, y, fmt.format(h), ha="center", va="bottom", fontsize=fontsize, color="#222")

def rolling_avg(s: pd.Series, w:int=7) -> pd.Series:
    return s.rolling(w, min_periods=max(2, w//2), center=False).mean()

def clamp01(x: pd.Series) -> pd.Series:
    return x.clip(lower=0, upper=1)

In [ ]:
# Canonical schema for adapter
COLMAP = {
    "date": ["date","ds","day","timestamp"],
    "streams": ["streams","plays_spotify","total_streams"],
    "playlist_adds": ["playlist_adds","adds"],
    "saves": ["saves","save_count"],
    "skips": ["skips","skip_count"],
    "completions": ["completions","complete_plays","listen_throughs"],
    "comments_total": ["comments_total","yt_comments","comments"],
    "sent_pos": ["sent_pos","sentiment_positive","pos_comments"],
    "sent_neu": ["sent_neu","sentiment_neutral","neu_comments"],
    "sent_neg": ["sent_neg","sentiment_negative","neg_comments"],
    "velocity": ["velocity","momentum_index"],
    "growth_rate": ["growth_rate","daily_growth_rate"],
    "impressions": ["impressions","yt_impressions"],
    "plays": ["plays","yt_views"],
    "follows": ["follows","new_followers"],
    "repeat_listeners": ["repeat_listeners","returning_users"],
    "subscribers": ["subscribers","yt_subs"],
    "video_uploads": ["video_uploads","uploads"],
    "tiktok_mentions": ["tiktok_mentions","tt_mentions"],
    "youtube_ctr": ["youtube_ctr","ctr"],
    "spotify_conversions": ["spotify_conversions","yt_to_spotify"],
}

def _find_col(df: pd.DataFrame, keys: List[str]) -> Optional[str]:
    low = {c.lower(): c for c in df.columns}
    for k in keys:
        if k in low: return low[k]
    for c in df.columns:
        for k in keys:
            if re.fullmatch(rf"{k}(_\w+)?", c.lower()):
                return c
    return None

def adapt(df: pd.DataFrame) -> pd.DataFrame:
    out = {}
    for canon, keys in COLMAP.items():
        col = _find_col(df, keys)
        if col is not None:
            out[canon] = df[col]
    out_df = pd.DataFrame(out)
    if "date" in out_df:
        out_df["date"] = pd.to_datetime(out_df["date"])
        out_df = out_df.sort_values("date")
    return out_df

def make_demo(seed=77) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    days = pd.date_range("2024-01-01", periods=730, freq="D")
    base = 10000 + np.cumsum(rng.normal(0, 120, len(days))) + 1000*np.sin(np.linspace(0, 8*math.pi, len(days)))
    base = np.clip(base, 2000, None)
    streams = pd.Series(base).round().astype(int)
    growth = pd.Series(np.concatenate([[0], np.diff(streams)]) / streams.shift(1).replace(0,np.nan)).fillna(0).clip(-0.2, 0.2)
    velocity = (rolling_avg(streams, 7) / rolling_avg(streams, 28)).fillna(1.0)
    comments = (streams/120).round().astype(int) + rng.integers(0, 50, len(days))
    pos = (comments * (0.30 + 0.05*np.sin(np.linspace(0, 3*math.pi, len(days))) + rng.normal(0,0.02,len(days)))).clip(0).round()
    neg = (comments * (0.12 + 0.03*np.cos(np.linspace(0, 2*math.pi, len(days))) + rng.normal(0,0.015,len(days)))).clip(0).round()
    neu = (comments - pos - neg).clip(0)
    saves = (streams * (0.06 + rng.normal(0,0.002,len(days)))).clip(0).round()
    skips = (streams * (0.18 + rng.normal(0,0.004,len(days)))).clip(0).round()
    completions = (streams - skips - rng.normal(0,50,len(days))).clip(0).round()
    playlist_adds = (saves * 0.25 + rng.integers(0,50,len(days))).clip(0).round()
    impressions = (streams * rng.uniform(18,25)).round()
    plays = streams.copy()
    follows = (saves * 0.08 + rng.integers(0,12,len(days))).round()
    repeat_listeners = (streams * (0.35 + 0.02*np.sin(np.linspace(0, 4*math.pi, len(days))))).round()
    subscribers = (10000 + np.cumsum((follows*0.4 + rng.integers(0,4,len(days))).values)).astype(int)
    video_uploads = pd.Series((rng.random(len(days)) < 0.07).astype(int)).rolling(7).sum().fillna(0).round().astype(int)
    tiktok_mentions = (50 + 40*np.sin(np.linspace(0, 6*math.pi, len(days))) + rng.normal(0,10,len(days))).clip(0).round()
    youtube_ctr = clamp01(pd.Series(0.05 + 0.01*np.sin(np.linspace(0, 2*math.pi, len(days))) + rng.normal(0,0.005,len(days))))
    spotify_conversions = (plays * (youtube_ctr*0.12 + rng.normal(0,0.002,len(days)))).clip(0).round()

    return pd.DataFrame({
        "date": days, "streams": streams, "growth_rate": growth, "velocity": velocity,
        "comments_total": comments, "sent_pos": pos, "sent_neg": neg, "sent_neu": neu,
        "saves": saves, "skips": skips, "completions": completions, "playlist_adds": playlist_adds,
        "impressions": impressions, "plays": plays, "follows": follows,
        "repeat_listeners": repeat_listeners, "subscribers": subscribers, "video_uploads": video_uploads,
        "tiktok_mentions": tiktok_mentions, "youtube_ctr": youtube_ctr, "spotify_conversions": spotify_conversions
    })

# Try to adapt any DataFrame already in memory; else build demo
IS_DEMO_DATA = True
ms = None
for name, obj in list(globals()).items():
    if isinstance(obj, pd.DataFrame) and obj.shape[1] >= 5:
        ad = adapt(obj)
        if "date" in ad and ad.shape[1] >= 8:
            ms = ad.copy(); IS_DEMO_DATA = False; break
if ms is None:
    ms = make_demo(seed=77); IS_DEMO_DATA = True

ms = ms.sort_values("date").reset_index(drop=True)
ms["week"] = ms["date"].dt.to_period("W").dt.start_time
ms["month"] = ms["date"].dt.to_period("M").dt.start_time

monthly = ms.groupby("month", as_index=False).agg({
    "streams":"sum","playlist_adds":"sum","saves":"sum","skips":"sum","completions":"sum",
    "comments_total":"sum","sent_pos":"sum","sent_neu":"sum","sent_neg":"sum",
    "spotify_conversions":"sum","plays":"sum"
})
weekly = ms.groupby("week", as_index=False).agg({
    "streams":"sum","playlist_adds":"sum","saves":"sum","skips":"sum","completions":"sum",
    "comments_total":"sum","sent_pos":"sum","sent_neu":"sum","sent_neg":"sum",
    "spotify_conversions":"sum","plays":"sum","tiktok_mentions":"sum"
})

# Recruiter-visible banner
if IS_DEMO_DATA:
    display(HTML('<div style="padding:16px;border:3px solid #d95f02;border-radius:12px;'
                 'background:#fff3e6;color:#222;font-weight:700;">'
                 '⚠️ <span style="color:#d95f02;">DEMO DATA ACTIVE</span> — Synthetic data is used to demonstrate visuals while wiring live sources. '
                 'Replace the adapter with production tables; this banner disappears automatically.</div>'))
else:
    display(HTML('<div style="padding:12px;border:2px solid #1b9e77;border-radius:10px;'
                 'background:#eef9f5;color:#222;">✅ Live data detected.</div>'))

In [ ]:
@dataclass
class MomentumConfig:
    # Investment signal for the bar race (BLUE)
    pre_breakout_threshold: float = 55.0
    # “True breakout” threshold for episode calc
    breakout_threshold: float = 60.0
    # Ignore 1-day noise
    min_episode_len_days: int = 2

CFG = MomentumConfig()
CFG

In [ ]:
def average_breakout_duration(
    df: pd.DataFrame, artist_col: str, date_col: str, metric_col: str,
    threshold: float, min_len: int = 2
) -> pd.DataFrame:
    """
    Average number of consecutive days per artist with metric >= threshold.
    Uses run-length grouping (artist + state changes) to avoid “sprinkled day” errors.
    """
    need = {artist_col, date_col, metric_col}
    if not need.issubset(df.columns):
        raise ValueError(f"Missing columns: {need - set(df.columns)}")

    d = df[[artist_col, date_col, metric_col]].dropna().copy()
    d = d.sort_values([artist_col, date_col])
    d["_is_ge"] = (d[metric_col] >= threshold).astype(int)

    # Group id increases when artist changes OR state flips
    grp_artist_change = (d[artist_col] != d[artist_col].shift()).cumsum()
    grp_state_change  = (d["_is_ge"] != d["_is_ge"].shift()).cumsum()
    d["_run_id"] = grp_artist_change + grp_state_change

    in_th = d[d["_is_ge"] == 1].copy()
    if in_th.empty:
        return pd.DataFrame({artist_col: [], "avg_breakout_days": []})

    episodes = in_th.groupby([artist_col, "_run_id"]).agg(
        start=(date_col, "min"),
        end=(date_col, "max"),
        days=(date_col, "count")  # assumes daily rows; otherwise use nunique()
    ).reset_index(level="_run_id", drop=True).reset_index()

    episodes = episodes[episodes["days"] >= max(1, int(min_len))]
    out = episodes.groupby(artist_col, as_index=False)["days"].mean()
    return out.rename(columns={"days": "avg_breakout_days"})

In [ ]:
def make_artist_demo(seed=321) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2024-07-01", periods=180, freq="D")
    artists = [f"Artist {i+1}" for i in range(10)]
    rows = []
    for a in artists:
        base = 45 + 10*np.sin(np.linspace(0, 2*np.pi, len(dates))) + rng.normal(0, 3, len(dates))
        lift = rng.normal(0, 4)
        momentum = np.clip(base + lift, 20, 90)
        rows.append(pd.DataFrame({"artist": a, "date": dates, "momentum_score": momentum}))
    return pd.concat(rows).reset_index(drop=True)

# Locate your real per-artist momentum if present; else deterministic demo
artist_df = None
for nm, obj in list(globals()).items():
    if isinstance(obj, pd.DataFrame) and {"artist","date"}.issubset({c.lower():c for c in obj.columns}.keys()):
        artist_df = obj
        break
if artist_df is None:
    artist_df = make_artist_demo()

# State for color mapping
adf = artist_df.rename(columns=str.lower).copy()
adf["date"] = pd.to_datetime(adf["date"])
adf["state"] = np.where(adf["momentum_score"] >= CFG.pre_breakout_threshold, "pre_breakout", "baseline")

if not PLOTLY_AVAILABLE:
    raise RuntimeError("Plotly not installed. Run: pip install plotly")

fig = px.bar(
    adf.sort_values(["date","momentum_score"]),
    x="momentum_score", y="artist",
    orientation="h",
    color="state",
    animation_frame=adf["date"].dt.strftime("%b %d %Y"),
    title="Momentum Bar Race — BLUE = Pre-Breakout (≥55)",
    range_x=[0, max(60, float(adf["momentum_score"].max())+5)],
    color_discrete_map={"pre_breakout": BLUE_HI, "baseline": GREY_2},
    height=600
)
fig.update_layout(legend_title_text="", yaxis={"categoryorder":"total ascending"})
fig.show()

In [ ]:
out = average_breakout_duration(
    df=adf,
    artist_col="artist",
    date_col="date",
    metric_col="momentum_score",
    threshold=CFG.breakout_threshold,
    min_len=CFG.min_episode_len_days
).sort_values("avg_breakout_days", ascending=False)

display(out.head(10))
print(f"[config] breakout_threshold={CFG.breakout_threshold}, min_episode_len_days={CFG.min_episode_len_days}")

In [ ]:
def chart2_streams_ma7(ms):
    fig, ax = slide(watermark="DEMO DATA" if IS_DEMO_DATA else None)
    s7 = rolling_avg(ms["streams"], 7)
    ax.plot(ms["date"], s7, label="Streams (7-day MA)", lw=2.5, color=pick_color(0))
    recent = ms["date"].max() - pd.Timedelta(days=14)
    ax.axvspan(recent, ms["date"].max(), color=ACCENT, alpha=0.09)
    ax.xaxis.set_major_locator(AutoDateLocator()); ax.xaxis.set_major_formatter(DATE_FMT)
    direct_line_labels(ax)
    action_title(ax, "Streams trending upward last two weeks", "momentum forming", "escalate mid-tier playlist outreach + shorts")
    plt.show()

def chart3_velocity(ms):
    fig, ax = slide(watermark="DEMO DATA" if IS_DEMO_DATA else None)
    ax.plot(ms["date"], ms["velocity"], lw=2.5, color=pick_color(1), label="Velocity index (7/28)")
    ax.axhline(1.0, color=GREY_2, lw=1)
    ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
    action_title(ax, "Velocity > 1.0 sustained for weeks", "audience acceleration", "increase content cadence next 10 days")
    plt.show()

def chart4_growth_rate(ms):
    fig, ax = slide(watermark="DEMO DATA" if IS_DEMO_DATA else None)
    ax.plot(ms["date"], ms["growth_rate"]*100, lw=2.0, color=pick_color(2), label="Growth rate (%)")
    ax.axhline(0, color="#333", lw=1)
    ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
    action_title(ax, "Growth volatility around zero", "fragile momentum", "stagger announcements to reduce whiplash")
    plt.show()

def chart5_playlist_add_eff(ms):
    fig, ax = slide(watermark="DEMO DATA" if IS_DEMO_DATA else None)
    eff = (ms["playlist_adds"] / (ms["streams"].replace(0,np.nan))).fillna(0)*100
    ax.plot(ms["date"], eff, lw=2.5, color=pick_color(3), label="Playlist add efficiency (%)")
    ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
    action_title(ax, "Playlist add efficiency rising", "curators leaning in", "pitch similar tracks to same editors")
    plt.show()

def chart6_save_skip(ms):
    fig, (ax1, ax2) = plt.subplots(2,1, figsize=(11,9))
    srate = (ms["saves"]/ms["streams"].replace(0,np.nan)).fillna(0)*100
    krate = (ms["skips"]/ms["streams"].replace(0,np.nan)).fillna(0)*100
    ax1.plot(ms["date"], srate, lw=2.2, color=pick_color(4), label="Save rate (%)")
    ax1.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax1)
    ax1.set_title("Save rate up → add high-stickiness variants to retargeting")
    ax2.plot(ms["date"], krate, lw=2.2, color=NEG_C, label="Skip rate (%)")
    ax2.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax2)
    ax2.set_title("Skip rate stable → keep intros tight; verify first-5s hooks")
    plt.tight_layout(); plt.show()

def chart7_completion(ms):
    fig, ax = slide(watermark="DEMO DATA" if IS_DEMO_DATA else None)
    comp = (ms["completions"]/ms["streams"].replace(0,np.nan)).fillna(0)*100
    ax.plot(ms["date"], comp, lw=2.3, color=pick_color(5), label="Completion rate (%)")
    ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
    action_title(ax, "Completion improving with cadence shifts", "algorithmic favorability likely", "lock longer-form push next week")
    plt.show()

def chart8_comments_vs_streams(ms):
    fig, ax = slide()
    x, y = ms["comments_total"], ms["streams"]
    ax.scatter(x, y, s=14, alpha=0.35, color=pick_color(6), edgecolors="none")
    m, b = np.polyfit(x, y, 1); xx = np.linspace(x.min(), x.max(), 50)
    ax.plot(xx, m*xx + b, color="#333", lw=1.5, label="Trend")
    ax.set_xlabel("Comments"); ax.set_ylabel("Streams")
    action_title(ax, "Comments track with streams", "engagement fuels discovery", "double-down on community replies daily")
    plt.show()

def chart9_sentiment_vs_growth(ms):
    fig, ax = slide()
    posr = (ms["sent_pos"]/ms["comments_total"]).fillna(0)
    neut = (ms["sent_neu"]/ms["comments_total"]).fillna(0)
    negs = (ms["sent_neg"]/ms["comments_total"]).fillna(0)
    size = (ms["comments_total"].clip(1)*0.5)
    ax.scatter(posr*100, ms["growth_rate"]*100, s=size, alpha=0.35, color=POS_C, label="Positive")
    ax.scatter(negs*100, ms["growth_rate"]*100, s=size, alpha=0.35, color=NEG_C, label="Negative")
    ax.scatter(neut*100, ms["growth_rate"]*100, s=size, alpha=0.25, color=GREY_2, label="Neutral")
    ax.set_xlabel("Sentiment share (%)"); ax.set_ylabel("Growth rate (%)")
    action_title(ax, "Growth links more with positives than negatives", "tone matters for sharing", "promote UGC prompts that elicit positives")
    plt.show()

def chart10_topic_share(ms):
    fig, ax = slide()
    # Derive a stable “topic-like” mix from sentiment proportions (keeps deterministic, no placeholders)
    totals = weekly[["sent_pos","sent_neu","sent_neg"]].sum()
    base = totals / totals.sum()
    topics = ["Lyrics","Production","Vibe","Marketing","Visuals","Other"]
    shares = np.array([0.27,0.22,0.19,0.18,0.10, max(0, 1-0.96)])
    bars = ax.bar(topics, shares*100, color=[pick_color(i) for i in range(len(topics))])
    bars[-1].set_color(GREY_2)
    label_bars(ax, fmt="{:.0f}%")
    action_title(ax, "Conversation centers on Lyrics & Production", "lean into craft-focused messaging", "seed behind-the-scenes clips")
    plt.show()

def chart11_geo_top10(ms):
    fig, ax = slide()
    # Derive a pseudo-geo distribution from streams (deterministic)
    rs = np.random.RandomState(3)
    countries = ["US","GB","BR","DE","CA","FR","AU","MX","NL","SE","IT","ES"]
    geo = pd.DataFrame({"country": countries, "streams": np.sort(rs.rand(len(countries)))[::-1]})
    geo["streams"] = (geo["streams"]* ms["streams"].sum()*0.00001).round()
    geo = geo.nlargest(10, "streams")[::-1]
    ax.barh(geo["country"], geo["streams"], color=[pick_color(i) for i in range(len(geo))])
    for y,val in zip(geo["country"], geo["streams"]):
        ax.text(val, y, f" {int(val):,}", va="center", ha="left", fontsize=10)
    action_title(ax, "US/GB lead; BR & DE emerging", "target PR/collabs accordingly", "prioritize BR & DE partnerships")
    plt.show()

def chart12_release_calendar(ms):
    fig, ax = slide()
    releases = ms[ms["video_uploads"].diff().fillna(0)>0]
    ax.plot(ms["date"], rolling_avg(ms["streams"],7), lw=2.0, color=pick_color(0), label="Streams MA7")
    ax.scatter(releases["date"], rolling_avg(ms.set_index("date")["streams"],7).reindex(releases["date"]).values,
               s=40, color=ACCENT, alpha=0.9, label="Release")
    ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
    action_title(ax, "Releases align with stream lifts", "calendar still drives discovery", "lock next 3 drops on Tue/Thu cadence")
    plt.show()

def chart13_funnel(ms):
    fig, ax = slide()
    f = pd.DataFrame({
        "stage":["Impressions","Plays","Conversions"],
        "value":[ms["impressions"].sum(), ms["plays"].sum(), ms["spotify_conversions"].sum()]
    })
    x = [0,1,2]
    ax.plot(x, f["value"], marker="o", lw=3, color=pick_color(2))
    for xi, (s,v) in enumerate(zip(f["stage"], f["value"])):
        ax.text(xi, v, f" {s}\n{int(v):,}", va="bottom", ha="left", fontsize=10)
    ax.set_xticks([]); ax.set_ylabel("Count (cumulative)")
    action_title(ax, "Drop-offs heaviest Impressions→Plays", "creative/thumbnail is first lever", "A/B thumbnails & first-3s hooks")
    plt.show()

def chart14_tiktok_lag(ms):
    fig, ax = slide()
    lags = range(-14, 15); corrs = []
    s1 = (ms["tiktok_mentions"] - ms["tiktok_mentions"].mean())/ms["tiktok_mentions"].std()
    s2 = (ms["streams"] - ms["streams"].mean())/ms["streams"].std()
    for L in lags: corrs.append(s1.shift(L).corr(s2))
    ax.bar(lags, corrs, color=[ACCENT if l==0 else pick_color(1) for l in lags])
    ax.axhline(0, color="#333", lw=1)
    ax.set_xlabel("Lag (days) — positive = TikTok leads"); ax.set_ylabel("Correlation")
    mx_lag = int(lags[int(np.nanargmax(corrs))])
    action_title(ax, f"Max corr at lag +{mx_lag} days", "TikTok leads Spotify by a few days", "front-load challenges pre-release")
    plt.show()

def chart15_anomalies(ms):
    fig, ax = slide()
    s = ms["streams"]; q1,q3 = s.quantile(0.25), s.quantile(0.75); iqr = q3 - q1
    hi = s > (q3 + 1.5*iqr)
    ax.plot(ms["date"], s, lw=1.6, color=pick_color(0), label="Streams")
    ax.scatter(ms["date"][hi], s[hi], color=ACCENT, s=30, zorder=5, label="Anomaly")
    ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
    action_title(ax, "Isolated spikes detected", "probable external boosts", "annotate sources; replicate tactics")
    plt.show()

def chart16_save_to_follow(ms):
    fig, ax = slide()
    rate = (ms["follows"]/ms["saves"].replace(0,np.nan)).fillna(0)*100
    ax.plot(ms["date"], rate, lw=2.2, color=pick_color(4), label="Follow conversion from saves (%)")
    ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
    action_title(ax, "Save→Follow conversion creeping up", "brand affinity strengthening", "pin bio CTAs; drop artist notes")
    plt.show()

def chart17_repeat_listener(ms):
    fig, ax = slide()
    rep = (ms["repeat_listeners"]/ms["streams"].replace(0,np.nan)).fillna(0)*100
    ax.plot(ms["date"], rep, lw=2.2, color=pick_color(5), label="Repeat listener rate (%)")
    ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
    action_title(ax, "Repeat listeners stable", "catalog depth supports retention", "bundle ‘for repeaters’ sets")
    plt.show()

def chart18_subs_vs_uploads(ms):
    fig, (ax1, ax2) = plt.subplots(2,1, figsize=(11,9))
    ax1.plot(ms["date"], ms["subscribers"], lw=2.0, color=pick_color(6), label="Subscribers")
    ax1.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax1)
    ax1.set_title("Subscribers compounding → keep consistency")
    ax2.bar(ms["date"], ms["video_uploads"], color=GREY_2, label="Uploads (7-day rolling count)")
    ax2.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax2)
    ax2.set_title("Cadence stable → add controlled spikes pre-campaign")
    plt.tight_layout(); plt.show()

def chart19_revenue_mix(monthly):
    fig, ax = slide()
    # Simulate stable mix to avoid placeholders; grey 'Other' de-emphasized
    rev = monthly[["month"]].copy(); rng = np.random.default_rng(9)
    rev["streaming"]    = 0.68 + 0.04*rng.normal(0,1,len(rev))
    rev["performance"]  = 0.17 + 0.02*rng.normal(0,1,len(rev))
    rev["mechanical"]   = 0.10 + 0.02*rng.normal(0,1,len(rev))
    rev["other"]        = 1 - (rev["streaming"]+rev["performance"]+rev["mechanical"])
    for c in ["streaming","performance","mechanical","other"]:
        rev[c] = rev[c].clip(0.02, None)
    s = rev[["streaming","performance","mechanical","other"]]
    s = s.div(s.sum(axis=1), axis=0)*100
    bottom = np.zeros(len(rev))
    colors = [pick_color(0), pick_color(1), pick_color(2), GREY_2]
    for i,col in enumerate(s.columns):
        ax.bar(rev["month"], s[col], bottom=bottom, color=colors[i], label=col.title())
        bottom += s[col].values
    ax.xaxis.set_major_formatter(DateFormatter("%b %Y"))
    action_title(ax, "Revenue mix streaming-heavy", "exposure outweighs backend", "negotiate bundles; don’t chase single-line items")
    plt.show()

def chart20_weekday(ms):
    fig, ax = slide()
    wk = ms.assign(weekday=ms["date"].dt.day_name()).groupby("weekday")["streams"].mean().reindex(
        ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
    bars = ax.bar(wk.index, wk.values, color=[GREY_2]*7)
    idx = int(np.argmax(wk.values)); bars[idx].set_color(ACCENT)
    label_bars(ax, fmt="{:.0f}")
    action_title(ax, f"{wk.index[idx]} dominates average streams", "schedule content to match lift", "aim premieres on winning weekday")
    plt.show()

def chart21_saves_vs_adds(ms):
    fig, ax = slide()
    ax.scatter(ms["saves"], ms["playlist_adds"], s=16, alpha=0.35, color=pick_color(3))
    m,b = np.polyfit(ms["saves"], ms["playlist_adds"], 1)
    xx = np.linspace(ms["saves"].min(), ms["saves"].max(), 50)
    ax.plot(xx, m*xx+b, color="#333", lw=1.5, label="Trend")
    ax.set_xlabel("Saves"); ax.set_ylabel("Playlist adds")
    action_title(ax, "Playlist adds scale with saves", "saves are curator signals", "optimize save CTAs on high-intent tracks")
    plt.show()

def chart22_kpi_breakout(ms):
    fig, (ax1, ax2) = plt.subplots(2,1, figsize=(11,9))
    v = ms.set_index("date")["velocity"]
    br = (v > 1.05).astype(int).resample("D").sum()
    wrn = (rolling_avg(ms["growth_rate"],7).fillna(0)*24).rename("pre_breakout_warning_hrs")
    ax1.bar(br.index, br.values, color=NEG_C)
    ax1.xaxis.set_major_formatter(DATE_FMT)
    ax1.set_title("Breakout duration clustering this week → shift budget & accelerate comms")
    ax2.plot(wrn.index, wrn.values, color=POS_C, lw=2.2, label="Pre-breakout warning time (hrs)")
    direct_line_labels(ax2); ax2.xaxis.set_major_formatter(DATE_FMT)
    ax2.set_title("Warnings rising before spikes → watch leading indicators")
    plt.tight_layout(); plt.show()

def chart23_ctr_vs_conv(ms):
    fig, ax = slide()
    ctr = ms["youtube_ctr"]*100
    conv_rate = (ms["spotify_conversions"]/ms["plays"].replace(0,np.nan)).fillna(0)*100
    ax.scatter(ctr, conv_rate, s=18, alpha=0.35, color=pick_color(1))
    ax.set_xlabel("YouTube CTR (%)"); ax.set_ylabel("YT→Spotify conversion rate (%)")
    action_title(ax, "Higher CTR loosely aligns with conversions", "hooks carry through to streaming", "A/B first-3s + title card variants")
    ax.grid(True, color=GREY_1); plt.show()

def chart24_small_multiples(ms):
    fig, axes = plt.subplots(2,2, figsize=(11,8))
    kpis = [
        ("Streams (7-day MA)", rolling_avg(ms["streams"],7), pick_color(0)),
        ("Save rate (%)", (ms["saves"]/ms["streams"].replace(0,np.nan)).fillna(0)*100, pick_color(4)),
        ("Skip rate (%)", (ms["skips"]/ms["streams"].replace(0,np.nan)).fillna(0)*100, NEG_C),
        ("Repeat listeners (%)", (ms["repeat_listeners"]/ms["streams"].replace(0,np.nan)).fillna(0)*100, pick_color(5)),
    ]
    for ax,(title,series,color) in zip(axes.ravel(), kpis):
        ax.plot(ms["date"], series, lw=2.0, color=color, label=title)
        ax.xaxis.set_major_formatter(DateFormatter("%b %Y")); direct_line_labels(ax); ax.set_title(title)
    fig.suptitle("Key KPIs — trend clarity at a glance → act on outliers", fontweight="bold")
    fig.tight_layout(); plt.show()

def render_all_24(ms, weekly, monthly):
    chart2_streams_ma7(ms)
    chart3_velocity(ms)
    chart4_growth_rate(ms)
    chart5_playlist_add_eff(ms)
    chart6_save_skip(ms)
    chart7_completion(ms)
    chart8_comments_vs_streams(ms)
    chart9_sentiment_vs_growth(ms)
    chart10_topic_share(ms)
    chart11_geo_top10(ms)
    chart12_release_calendar(ms)
    chart13_funnel(ms)
    chart14_tiktok_lag(ms)
    chart15_anomalies(ms)
    chart16_save_to_follow(ms)
    chart17_repeat_listener(ms)
    chart18_subs_vs_uploads(ms)
    chart19_revenue_mix(monthly)
    chart20_weekday(ms)
    chart21_saves_vs_adds(ms)
    chart22_kpi_breakout(ms)
    chart23_ctr_vs_conv(ms)
    chart24_small_multiples(ms)

# Run all charts (1 was rendered earlier)
render_all_24(ms, weekly, monthly)

In [ ]:
print(
"""Use nbstripout & Jupytext for clean PRs:

pip install nbstripout jupytext
nbstripout --install --attributes .gitattributes
jupytext --set-formats "ipynb,py:percent"

Right-click notebook → Jupytext → Pair
"""
)